In [ ]:
# ============================================================
# COLAB USERS ONLY — Uncomment and run this cell
# Local VS Code users: SKIP this cell
# ============================================================

# !pip install langchain langchain-groq langchain-community python-dotenv duckduckgo-search -q

# from google.colab import userdata
# import os
# os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
# print("Colab setup complete!")

# Day 5: LangChain Agents & Tools — Deep Dive

**Agentic AI Hands-On Course** | Dr. Kanthi Kiran Sirra | Sr. AI Engineer

**What you'll build today:**
1. Understand AgentExecutor internals — what happens inside `.invoke()`
2. Master custom tool design — the 5 golden rules
3. Implement 3 levels of error handling
4. Use `max_iterations` and `handle_parsing_errors` for safety
5. Build a multi-tool research assistant (5+ tools)
6. Debug agents using verbose mode

**Deliverable:** Multi-tool research assistant with error handling

---

## 0. Setup

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

groq_key = os.getenv("GROQ_API_KEY", "")
print(f"Groq API Key: {'Loaded' if len(groq_key) > 10 else 'MISSING'}")

In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0, max_tokens=500)

# Quick test
response = llm.invoke("Say 'Day 5 ready!' in exactly 3 words.")
print(response.content)

---
## 1. AgentExecutor Internals — What happens inside `.invoke()`

Yesterday you used `AgentExecutor` as a black box. Today we open it up.

The loop inside AgentExecutor:
```
1. Format prompt (system + user input + scratchpad)
2. Call LLM (reads prompt + tool descriptions)
3a. If tool call → execute tool → add result to scratchpad → GO TO 2
3b. If final answer → return it → loop ends
4. If max_iterations exceeded → force stop
```

Let's build a **manual agent loop** to see this in action:

In [ ]:
from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.agents import create_tool_calling_agent, AgentExecutor

# A simple tool to demonstrate the loop
@tool
def multiply(a: int, b: int) -> str:
    """Multiplies two numbers together.
    
    Args:
        a: First number
        b: Second number
    """
    return str(int(a) * int(b))

# Create agent with verbose=True to see the loop
agent_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Use tools when needed. Be concise."),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

agent = create_tool_calling_agent(llm, [multiply], agent_prompt)
executor = AgentExecutor(agent=agent, tools=[multiply], verbose=True)

# Watch the loop in action
print("=== Single tool call (1 iteration) ===")
result = executor.invoke({"input": "What is 15 times 23?"})
print(f"\nFinal: {result['output']}")

### 1.1 The agent_scratchpad — agent's working memory

The `MessagesPlaceholder("agent_scratchpad")` in the prompt is where AgentExecutor inserts tool call history.

- **Before first tool call:** scratchpad is empty `[]`
- **After calling multiply(15, 23):** scratchpad contains the AIMessage with the tool call + ToolMessage with result `"345"`
- **On next iteration:** LLM reads the scratchpad and knows the result already

Without this scratchpad, the agent would have amnesia between iterations — calling the same tool forever.

---

## 2. Custom Tool Design — The 5 Golden Rules

A good tool follows 5 rules:
1. **Docstring is everything** — LLM reads it to decide when to use the tool
2. **Always return strings** — LLM processes text
3. **Never raise exceptions** — return error strings so the agent can recover
4. **Include context in output** — don't return `9310`, return `245 * 38 = 9,310`
5. **Type hints on every argument** — agent uses them to pass correct data types

Let's build 5 production-quality tools:

In [ ]:
import datetime

@tool
def calculate(expression: str) -> str:
    """Evaluates a mathematical expression and returns the result.
    Use this for any math: arithmetic, percentages, compound interest, etc.
    
    Args:
        expression: A mathematical expression like '245 * 38' or '50000 * 1.085 ** 5'
    """
    try:
        result = eval(expression)
        # Rule 4: Include context in output
        return f"{expression} = {result:,.2f}" if isinstance(result, float) else f"{expression} = {result:,}"
    except Exception as e:
        # Rule 3: Never raise, return error string
        return f"Error evaluating '{expression}': {e}. Please check the syntax."

# Test
print(calculate.invoke("245 * 38"))
print(calculate.invoke("50000 * 1.085 ** 5"))
print(calculate.invoke("invalid expression"))  # error handling

In [ ]:
@tool
def search_web(query: str) -> str:
    """Searches the web for current information using DuckDuckGo.
    Use this when you need up-to-date information not in your training data.
    
    Args:
        query: The search query, e.g. 'India GDP 2025' or 'latest AI news'
    """
    try:
        from duckduckgo_search import DDGS
        results = DDGS().text(query, max_results=3)
        if not results:
            return f"No results found for '{query}'. Try a different search query."
        output = []
        for i, r in enumerate(results, 1):
            output.append(f"{i}. {r['title']}: {r['body'][:200]}")
        return "\n\n".join(output)
    except Exception as e:
        return f"Search failed for '{query}': {e}. Try a simpler query."

# Test
print(search_web.invoke("Python programming language")[:300], "...")

In [ ]:
@tool
def text_analyzer(text: str) -> str:
    """Analyzes text and returns word count, character count, sentence count, and readability.
    Use this when someone asks to count words, analyze text length, or check readability.
    
    Args:
        text: The text to analyze
    """
    try:
        words = text.split()
        word_count = len(words)
        char_count = len(text)
        sentence_count = text.count('.') + text.count('!') + text.count('?') or 1
        avg_word_len = sum(len(w) for w in words) / max(word_count, 1)
        avg_sent_len = word_count / max(sentence_count, 1)
        
        if avg_sent_len < 12:
            readability = "Easy"
        elif avg_sent_len < 20:
            readability = "Moderate"
        else:
            readability = "Complex"
        
        return (f"Words: {word_count} | Characters: {char_count} | "
                f"Sentences: {sentence_count} | "
                f"Avg word length: {avg_word_len:.1f} chars | "
                f"Avg sentence length: {avg_sent_len:.1f} words | "
                f"Readability: {readability}")
    except Exception as e:
        return f"Error analyzing text: {e}"

# Test
print(text_analyzer.invoke("AI agents are transforming software. They can reason and use tools. This changes everything."))

In [ ]:
@tool
def convert_units(value: float, from_unit: str, to_unit: str) -> str:
    """Converts between common units of measurement.
    Supports: km/miles, kg/pounds, liters/gallons, cm/inches, celsius/fahrenheit
    
    Args:
        value: The numeric value to convert
        from_unit: Source unit (km, miles, kg, pounds, liters, gallons, cm, inches, celsius, fahrenheit)
        to_unit: Target unit
    """
    try:
        value = float(value)
        conversions = {
            ('km', 'miles'): lambda v: v * 0.621371,
            ('miles', 'km'): lambda v: v * 1.60934,
            ('kg', 'pounds'): lambda v: v * 2.20462,
            ('pounds', 'kg'): lambda v: v * 0.453592,
            ('liters', 'gallons'): lambda v: v * 0.264172,
            ('gallons', 'liters'): lambda v: v * 3.78541,
            ('cm', 'inches'): lambda v: v * 0.393701,
            ('inches', 'cm'): lambda v: v * 2.54,
            ('celsius', 'fahrenheit'): lambda v: v * 9/5 + 32,
            ('fahrenheit', 'celsius'): lambda v: (v - 32) * 5/9,
        }
        
        key = (from_unit.lower(), to_unit.lower())
        if key not in conversions:
            supported = ', '.join([f"{a}->{b}" for a, b in conversions.keys()])
            return f"Unsupported conversion: {from_unit} to {to_unit}. Supported: {supported}"
        
        result = conversions[key](value)
        return f"{value} {from_unit} = {result:.4f} {to_unit}"
    except Exception as e:
        return f"Conversion error: {e}"

# Test
print(convert_units.invoke({"value": 10, "from_unit": "km", "to_unit": "miles"}))
print(convert_units.invoke({"value": 37, "from_unit": "celsius", "to_unit": "fahrenheit"}))
print(convert_units.invoke({"value": 5, "from_unit": "km", "to_unit": "pounds"}))  # error case

In [ ]:
@tool
def get_date_info(date_string: str) -> str:
    """Gets information about a specific date: day of week, days until/since today, and day of year.
    Use this for date-related questions.
    
    Args:
        date_string: A date in YYYY-MM-DD format, e.g. '2025-08-15'
    """
    try:
        date = datetime.datetime.strptime(date_string, "%Y-%m-%d")
        today = datetime.datetime.now()
        day_name = date.strftime("%A")
        diff = (date - today).days
        
        if diff > 0:
            time_info = f"{diff} days from now"
        elif diff < 0:
            time_info = f"{abs(diff)} days ago"
        else:
            time_info = "Today!"
        
        day_of_year = date.timetuple().tm_yday
        return f"{date.strftime('%B %d, %Y')} ({day_name}) | {time_info} | Day {day_of_year} of the year"
    except ValueError:
        return f"Invalid date format: '{date_string}'. Please use YYYY-MM-DD (e.g., '2025-08-15')."

# Test
print(get_date_info.invoke("1947-08-15"))
print(get_date_info.invoke("bad-date"))  # error case

**Notice the pattern in every tool:**
1. Clear docstring with "Use this when..." guidance
2. Args documented with examples
3. `try/except` wrapping the entire body
4. Contextual output (not just raw numbers)
5. Type hints on every parameter

---

## 3. Error Handling — 3 Levels of Protection

### Level 1: Inside the tool (already done above)
Every tool has `try/except`. Errors return strings, not exceptions.

### Level 2: On the AgentExecutor

In [ ]:
# All 5 tools together
all_tools = [calculate, search_web, text_analyzer, convert_units, get_date_info]

agent_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful research assistant. Use the available tools when needed. Be concise and precise."),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

agent = create_tool_calling_agent(llm, all_tools, agent_prompt)

# Level 2: Safety controls on the executor
research_agent = AgentExecutor(
    agent=agent,
    tools=all_tools,
    verbose=True,
    max_iterations=5,            # stop after 5 tool calls (prevents infinite loops)
    handle_parsing_errors=True,  # recover from malformed LLM JSON output
)

print(f"Research agent ready with {len(all_tools)} tools: {[t.name for t in all_tools]}")
print(f"Safety: max_iterations=5, handle_parsing_errors=True")

### Level 3: Around the invocation

In [ ]:
def safe_ask(agent_executor, question):
    """Level 3: Wraps the agent call in try/except for graceful failure."""
    try:
        result = agent_executor.invoke({"input": question})
        return result["output"]
    except Exception as e:
        return f"Sorry, I encountered an error processing your request: {e}"

# Test with a normal question
answer = safe_ask(research_agent, "What is 123 * 456?")
print(f"Answer: {answer}")

---
## 4. Testing the Research Agent — Tool Selection

Same agent, different questions — watch which tool it picks automatically.

In [ ]:
# Query 1: Should use calculate
print("=" * 60)
print("Query 1: Math")
print("=" * 60)
print(safe_ask(research_agent, "If I invest 50,000 rupees at 8.5% compound interest for 5 years, how much will I have?"))

In [ ]:
# Query 2: Should use search_web
print("=" * 60)
print("Query 2: Current information")
print("=" * 60)
print(safe_ask(research_agent, "What is the current population of Hyderabad?"))

In [ ]:
# Query 3: Should use text_analyzer
print("=" * 60)
print("Query 3: Text analysis")
print("=" * 60)
print(safe_ask(research_agent, 
    "Analyze this text: Artificial intelligence agents represent a paradigm shift in software development. "
    "They can autonomously reason, plan, and execute complex tasks using external tools."))

In [ ]:
# Query 4: Should use convert_units
print("=" * 60)
print("Query 4: Unit conversion")
print("=" * 60)
print(safe_ask(research_agent, "I ran 10 km today. How many miles is that?"))

In [ ]:
# Query 5: Should use get_date_info
print("=" * 60)
print("Query 5: Date question")
print("=" * 60)
print(safe_ask(research_agent, "What day of the week was India's independence day — August 15, 1947?"))

In [ ]:
# Query 6: Should NOT use any tool — direct answer
print("=" * 60)
print("Query 6: General knowledge (no tool needed)")
print("=" * 60)
print(safe_ask(research_agent, "What is the capital of Japan?"))

**6 queries, 5 different tools (+ 1 direct answer).** The agent read all 5 tool docstrings and picked the right one each time. This is the power of good docstrings.

---

## 5. Tool Chaining — Multiple Tools for One Question

The real power of agents: one question that requires **multiple tool calls in sequence**.

In [ ]:
# This question needs: search + calculate
print("=" * 60)
print("Multi-tool query: Search + Calculate")
print("=" * 60)
print(safe_ask(research_agent,
    "Search for India's GDP in US dollars, then calculate what 15% of that amount is."
))

In [ ]:
# This needs: calculate + convert_units
print("=" * 60)
print("Multi-tool query: Calculate + Convert")
print("=" * 60)
print(safe_ask(research_agent,
    "I drove 120 km to work and 120 km back. My car uses 8 liters per 100 km. "
    "How many gallons of fuel did I use today?"
))

**Watch the verbose output above.** The agent:
1. Called the first tool and got a result
2. Read that result in the scratchpad
3. Decided it needed another tool call
4. Used the first result as input for the second tool
5. Combined everything into the final answer

This is **tool chaining** — the agent's ability to orchestrate multiple tools for a single question.

---

## 6. Verbose Debugging — Reading Agent Logs

When things go wrong, verbose output is your best friend. Here's what to look for:

| Log Line | What It Means | If Wrong, Fix... |
|---|---|---|
| `Thought:` | Agent's reasoning | System prompt |
| `Action:` | Which tool it chose | Tool docstrings |
| `Action Input:` | Arguments passed | Arg descriptions in docstring |
| `Observation:` | Tool's return value | Tool function code |
| `Final Answer:` | The conclusion | Prompt + tool quality |

### 6.1 Demonstrating a bad docstring

In [ ]:
# BAD TOOL — vague docstring
@tool
def process_data(data: str) -> str:
    """Processes data.
    
    Args:
        data: The data to process
    """
    return f"Processed: {len(data)} characters"

# GOOD TOOL — same function, clear docstring
@tool
def count_characters(text: str) -> str:
    """Counts the number of characters in the given text.
    Use this when someone asks about text length or character count.
    
    Args:
        text: The text to measure
    """
    return f"The text contains {len(text)} characters."

print("Bad tool — the agent won't know when to use 'process_data'")
print(f"  Name: {process_data.name}")
print(f"  Desc: {process_data.description}")
print()
print("Good tool — the agent knows exactly when to use 'count_characters'")
print(f"  Name: {count_characters.name}")
print(f"  Desc: {count_characters.description}")

---
## 7. Agent Types — tool_calling vs react

LangChain offers two agent types:

| | `create_tool_calling_agent` | `create_react_agent` |
|---|---|---|
| **How it works** | LLM generates JSON for tool calls | LLM generates text (Thought/Action/...) |
| **Reliability** | Higher — JSON schema constrains output | Lower — free text can be malformed |
| **Works with** | LLMs with function calling (Groq, OpenAI, Gemini) | ANY LLM |
| **Use when** | Your LLM supports function calling (most do) | Ollama local models without tool support |

**Rule:** Use `create_tool_calling_agent` unless your LLM doesn't support function calling.

We use `create_tool_calling_agent` throughout this course because Groq supports it.

---

## 8. Complete Application — Research + Structured Output

Let's combine the agent with a structured output chain for a real-world pattern:

In [ ]:
import json
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Step 1: Agent researches
raw_research = safe_ask(research_agent, "Search for the latest trends in AI agents in 2026")
print("=== Raw Research ===")
print(raw_research[:400], "...")
print()

# Step 2: Chain structures the output
structure_prompt = ChatPromptTemplate.from_messages([
    ("system", "Convert research findings into structured JSON. Respond ONLY with valid JSON, no markdown, no backticks."),
    ("human", "Structure these findings with keys: topic (string), summary (string), key_trends (list of 3 strings), impact (string).\n\nFindings: {findings}")
])

structure_chain = structure_prompt | llm | JsonOutputParser()

try:
    structured = structure_chain.invoke({"findings": raw_research})
    print("=== Structured Output ===")
    print(json.dumps(structured, indent=2))
except Exception as e:
    print(f"Structuring failed: {e}")
    print("Raw research is still available above.")

---
## 9. Your Turn — Extend the Research Agent

**Challenge (10 minutes):** Add a new custom tool to the research agent.

**Ideas:**
- `calculate_bmi(weight_kg, height_cm)` — BMI + health category
- `calculate_emi(principal, rate, months)` — loan EMI calculator
- `password_strength(password)` — checks password quality
- `text_to_hashtags(text)` — generates hashtags from text

**Requirements:**
1. Follow the 5 golden rules
2. Add it to `all_tools` and recreate the agent
3. Test with 2 queries — one for your tool, one for an existing tool

In [ ]:
# YOUR CODE HERE

# Step 1: Define your tool
# @tool
# def my_tool(param1: type, param2: type) -> str:
#     """Clear description. Use this when...
#     
#     Args:
#         param1: Description with example
#         param2: Description with example
#     """
#     try:
#         # your logic
#         return f"contextual result"
#     except Exception as e:
#         return f"Error: {e}"

# Step 2: Add to tools and recreate agent
# extended_tools = all_tools + [my_tool]
# agent = create_tool_calling_agent(llm, extended_tools, agent_prompt)
# extended_agent = AgentExecutor(
#     agent=agent, tools=extended_tools,
#     verbose=True, max_iterations=5, handle_parsing_errors=True
# )

# Step 3: Test
# print(safe_ask(extended_agent, "Your test query here"))

---
## 10. Day 5 Wrap-up

### What you built today:
- Understood AgentExecutor internals — the loop, the scratchpad
- Mastered 5 golden rules for custom tool design
- Implemented 3 levels of error handling (tool / executor / invocation)
- Used `max_iterations` and `handle_parsing_errors` for safety
- Built a 5-tool research assistant
- Demonstrated tool chaining (multi-tool per query)
- Learned to debug agents with verbose output
- Compared tool_calling_agent vs react_agent

### The error handling checklist:
```python
# Level 1: Inside every tool
try: ... except: return f"Error: {e}"

# Level 2: On the executor
AgentExecutor(max_iterations=5, handle_parsing_errors=True)

# Level 3: Around the call
try: agent.invoke(...) except: return fallback
```

### Tomorrow — Day 6: Multi-Agent Systems with CrewAI
Multiple agents working together — a Researcher, Writer, and Editor — each with their own role, goal, and tools. This is where AI gets really powerful.

---

### Resources:
- [LangChain agents how-to](https://python.langchain.com/docs/how_to/#agents)
- [LangChain custom tools](https://python.langchain.com/docs/how_to/custom_tools/)
- [AgentExecutor reference](https://python.langchain.com/api_reference/langchain/agents/langchain.agents.agent.AgentExecutor.html)